# Efficient NLP Dataset Extractor (The "New Approach")

This notebook mathematically solves the `"Provide Solution"` class imbalance bias diagnosed in the RL evaluation. 
Instead of relying on manual keywords, it leverages a tiny, highly-distilled HuggingFace model (`all-MiniLM-L6-v2`) running efficiently via **SentenceTransformers**. We compute the `Cosine Similarity` between the semantic meaning of the dataset text and our target Categories.

In [ ]:
# Install required deep learning libraries directly from the notebook
# !pip install -q transformers torch pandas tqdm scikit-learn

import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm.auto import tqdm
import csv

# Load the robust BART model for Zero-Shot Classification
# device=-1 ensures it runs on your Ryzen 7 CPU instead of trying to use the limited 2GB iGPU
print("Loading facebook/bart-large-mnli into memory (CPU Mode)...")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)

# Define labels
sentiment_labels = [
    "angry frustrated complaint", 
    "neutral statement", 
    "happy grateful appreciation"
]

action_labels = [
    "asking for information or clarifying details",
    "providing a distinct solution or technical answer",
    "apologizing or offering affective empathy repair",
    "escalating to support team or email",
    "closing the conversation goodbye",
    "giving a proactive system checking update",
    "setting expectations or asking to wait"
]



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\Devyansh\AppData\Roaming\Python\Python312\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(



[TensorFlow DLL Diagnostic] Analyzing: C:\Users\Devyansh\AppData\Roaming\Python\Python312\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "C:\Users\Devyansh\AppData\Roaming\Python\Python312\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

## 1. Zero-Shot Inference Engine\nThis cell leverages the NLI (Natural Language Inference) logic of BART to classify text.

In [2]:
def classify_text(text, candidate_labels):
    text = str(text).strip()[:400]
    if len(text) < 2: return 1 # Default to Neutral

    # BART Zero-Shot classification
    result = classifier(text, candidate_labels, multi_label=False)
    
    # result['labels'][0] is the top prediction. We find its index in our original list.
    top_label = result['labels'][0]
    return candidate_labels.index(top_label)

print("Testing BART Logic...")
test_customer = "I've been trying to log in for 3 hours and the system keeps dropping my connection!"
print(f"Predicted Sentiment Index: {classify_text(test_customer, sentiment_labels)}")

test_agent = "I am so sorry about this downtime. My team is looking into the server right now."
print(f"Predicted Action Index: {classify_text(test_agent, action_labels)}")


Testing BART Logic...


NameError: name 'sentiment_labels' is not defined

## 2. Generate Dataset\nProcessing turns. Note: BART is slower than MiniLM but significantly more accurate for RL state-action alignment.

In [ ]:
# You can adjust 'nrows=None' to run the entire dataset. 
# We sample 5,000 here to ensure it finishes quickly for testing. Let it run overnight for the full 8 million Reddit rows.

TWITTER_PATH = r"D:\SEM_6\RL\Project\my_local_twitter\twcs\twcs.csv"
df_tw = pd.read_csv(TWITTER_PATH, nrows=5000)

outbound = df_tw[df_tw['inbound'] == False].dropna(subset=['in_response_to_tweet_id'])
outbound_dict = {row['in_response_to_tweet_id']: row for _, row in outbound.iterrows()}
inbound_dict = {row['in_response_to_tweet_id']: row for _, row in df_tw.iterrows()}

print("Extracting Semantically Aligned NLP tuples using BART...")
nlp_tuples = []

for _, initial_tweet in tqdm(df_tw[df_tw['inbound'] == True].iterrows(), total=len(df_tw[df_tw['inbound'] == True])):
    tid = initial_tweet['tweet_id']
    
    if tid in outbound_dict:
        agent_reply = outbound_dict[tid]
        
        # 1. NLP Classify initial state
        state_idx = classify_text(initial_tweet['text'], sentiment_labels)
        
        # 2. NLP Classify agent action
        action_idx = classify_text(agent_reply['text'], action_labels)
        
        if agent_reply['tweet_id'] in inbound_dict:
            customer_reply = inbound_dict[agent_reply['tweet_id']]
            
            # 3. NLP Classify next state
            next_state_idx = classify_text(customer_reply['text'], sentiment_labels)
            
            # Pure Reward Function
            reward = 10.0 if next_state_idx == 2 else (-10.0 if next_state_idx == 0 else -2.0)
            
            nlp_tuples.append((state_idx, action_idx, reward, next_state_idx, True))

# Save the pristine high-fidelity NLP dataset
with open("nlp_offline_dataset_twitter.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["state", "action", "reward", "next_state", "done"])
    writer.writerows(nlp_tuples)
    
print(f"\n> Saved {len(nlp_tuples)} Pristine NLP transitions using BART.")
print("> Run Train_Offline_RL.ipynb on 'nlp_offline_dataset_twitter.csv' to see the bias vanish!")
